# Validation of the weighted pathway-centered abstraction of Reactome

Find a set a co-expressed genes and evaluate if they are closer than randomly expected in the pathway-centered abstraction of Reactome

In [23]:
import pandas as pd
import numpy as np
import subprocess
import random
from itertools import combinations  
import networkx as nx

1. Load weighted pathway abstraction

In [2]:
is_a_component_of = pd.read_csv("../Results/ReactomeHomoSapiens94WeightedAbstractions/Weights_IsAComponentOf/ReactomeHomoSapiens94_IsAComponentOf_Resnik.csv", sep=",", header=0)
print(is_a_component_of.head())

next_step_pathway = pd.read_csv("../Results/ReactomeHomoSapiens94WeightedAbstractions/Weights_NextStepPathway/ReactomeHomoSapiens94_NextStepPathway_ERcontent.csv", sep=",", header=0)
print(next_step_pathway.head())

global_abstraction = pd.concat([is_a_component_of, next_step_pathway], axis=0)
global_abstraction.to_csv("../Results/ReactomeHomoSapiens94WeightedAbstractions/ReactomeHomoSapiens94_WeightedPathwayAbstraction.csv", sep=",", index=False)

          source                 interaction         target    weight
0  R-HSA-8877627  abstraction:IsAComponentOf  R-HSA-6806667  4.810769
1  R-HSA-6806664  abstraction:IsAComponentOf  R-HSA-6806667  4.810769
2   R-HSA-975634  abstraction:IsAComponentOf  R-HSA-6806667  4.810769
3  R-HSA-1296041  abstraction:IsAComponentOf  R-HSA-1296059  6.336826
4   R-HSA-190241  abstraction:IsAComponentOf  R-HSA-5654738  4.185064
          source                  interaction         target    weight
0  R-HSA-1474151  abstraction:NextStepPathway  R-HSA-9009391  5.369342
1  R-HSA-1474151  abstraction:NextStepPathway   R-HSA-203615  5.731457
2  R-HSA-1474151  abstraction:NextStepPathway  R-HSA-5218920  5.731457
3   R-HSA-434313  abstraction:NextStepPathway   R-HSA-200425  6.304803
4   R-HSA-174411  abstraction:NextStepPathway   R-HSA-174414  5.910148


2. Load gene set and convert to UniprotKB identifiers

In [3]:
raw_data = pd.read_table("../CancerData/MSigDB_Coexp_2005.txt")
print(raw_data.head())

coexp_genes = list(raw_data["GeneSym"])
print(coexp_genes)

preprocessed_data = pd.DataFrame(columns=['GeneName'])
preprocessed_data['GeneName'] = coexp_genes
preprocessed_data.to_csv('../CancerData/MSigDB_Coexp_gene_list.csv', sep=',', header=None, index=False)
print(preprocessed_data.head())

  GeneSym  NA  GeneID
0    ANK1  na     286
1    SPTB  na    6710
2    TAL1  na    6886
3  MAP2K3  na    5606
4  BNIP3L  na     665
['ANK1', 'SPTB', 'TAL1', 'MAP2K3', 'BNIP3L', 'SPTA1', 'CDC27', 'KAT2B', 'PRDX2', 'AATF', 'RPA2', 'CDK2', 'DDB1', 'PRKAG1', 'PHB', 'USP5', 'BECN1', 'RAF1', 'TERF2IP', 'CSNK1D', 'SS18', 'TPR', 'BMI1', 'EIF4E', 'RFC1', 'ACP1', 'CCNI', 'SART1', 'ANP32B', 'XRCC6', 'UBE2I', 'DEK', 'CTBP1', 'HDAC1', 'ERH', 'DAP3', 'EIF3E', 'FBL', 'RAN', 'ACTG1', 'NPM1', 'TPT1', 'NME2', 'EIF4A2', 'JUND', 'ST13', 'APEX1', 'PSME1', 'PFN1', 'CBFB', 'SNRNP70', 'AP3D1', 'PAPSS1', 'CUL1', 'PPP2CA', 'SEPT7', 'TERF1', 'BUB3', 'RAD23A', 'EIF3I', 'GNB1', 'XRCC5', 'HDAC2', 'EI24', 'DNMT1', 'PPP1CC', 'RFC4', 'RRM1', 'PRKDC', 'RAD54L', 'CDC16', 'DEAF1', 'CSNK2B', 'SOD1', 'HAT1', 'MAP2K2', 'PRDX3', 'UNG', 'GMPS', 'PTPN11', 'ELAC2', 'KPNB1', 'TDG', 'MCM5', 'SMC1A', 'MSH2', 'RPA1', 'MLH1', 'PA2G4', 'MSH6', 'AP2M1', 'PPP1CA', 'SKP1', 'PSMC1', 'RAD21', 'ATOX1', 'RAD23B', 'PSMC2', 'RAB5A', 'MBD4', '

In [4]:
mapping_uniprot = pd.read_csv('../CancerData/MSigDB_Coexp_uniprot_list.tsv', sep="\t", header=0)

uniprot_ids = list()
for index, row in mapping_uniprot.iterrows():
    if row[2] == "reviewed":
        uniprot_ids.append(row[1])
    
print(uniprot_ids)

/tmp/ipykernel_52164/3769777933.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[2] == "reviewed":
/tmp/ipykernel_52164/3769777933.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  uniprot_ids.append(row[1])


['P16157', 'P11277', 'P17542', 'P46734', 'O60238', 'P02549', 'P30260', 'Q92831', 'P32119', 'Q9NY61', 'P15927', 'P24941', 'Q16531', 'P54619', 'P35232', 'P45974', 'Q14457', 'P04049', 'Q9NYB0', 'P48730', 'Q15532', 'P12270', 'P35226', 'P06730', 'P35251', 'P41440', 'P24666', 'Q14094', 'O43290', 'Q92688', 'P12956', 'P63279', 'P35659', 'Q13363', 'Q13547', 'P84090', 'O95886', 'P51398', 'P60228', 'P22087', 'P62826', 'P63261', 'P06748', 'P13693', 'P22392', 'Q14240', 'P17535', 'P50502', 'P27695', 'Q06323', 'P07737', 'Q13951', 'P08621', 'O14617', 'O43252', 'Q13616', 'P67775', 'Q16181', 'P54274', 'O43684', 'P54725', 'Q13347', 'P62873', 'P13010', 'Q92769', 'O14681', 'P26358', 'P36873', 'P35249', 'P23921', 'P78527', 'P46100', 'Q92698', 'Q13042', 'O75398', 'P67870', 'P00441', 'O14929', 'P36507', 'P30048', 'P13051', 'P49915', 'Q06124', 'Q9BQ52', 'Q14974', 'Q13569', 'P33992', 'Q14683', 'P43246', 'P27694', 'P40692', 'Q9UQ80', 'P52701', 'Q96CW1', 'P62136', 'P63208', 'P62191', 'O60216', 'O00244', 'P54727',

3. Get UniprotKB IDs associated to Reactome pathways

In [20]:
# Map the gene set to the corresponding pathways in the abstraction
uniprot_per_pathway = pd.read_csv("../Results/ReactomeHomoSapiens94UtilityFiles/ReactomeHomoSapiens94_UpIdPerPathway.csv", sep=",", header=0)

dico_up_id_per_pathway = dict()
counter = 0
for index, row in uniprot_per_pathway.iterrows():
    if not row[0] in dico_up_id_per_pathway.keys() and not pd.isna(uniprot_per_pathway.iloc[counter,1]) :
        dico_up_id_per_pathway[row[0]] = [row[1]]
    if row[0] in dico_up_id_per_pathway.keys() and not pd.isna(uniprot_per_pathway.iloc[counter,1]):
        dico_up_id_per_pathway[row[0]] += [row[1]]
    counter += 1

print(dico_up_id_per_pathway)

/tmp/ipykernel_52164/2584543280.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not row[0] in dico_up_id_per_pathway.keys() and not pd.isna(uniprot_per_pathway.iloc[counter,1]) :
/tmp/ipykernel_52164/2584543280.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[0] in dico_up_id_per_pathway.keys() and not pd.isna(uniprot_per_pathway.iloc[counter,1]):
/tmp/ipykernel_52164/2584543280.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `se

{'R-HSA-9027276': ['P01588', 'P01588', 'P07948', 'O60674', 'P19235', 'Q9Y4H2', 'Q13480'], 'R-HSA-8964043': ['P98155', 'P98155', 'Q8NBP7', 'Q00610', 'P09496', 'P53680', 'Q96CW1', 'P63010', 'Q8WY64', 'P01130', 'P04114', 'P61916', 'O15118', 'P38571', 'Q6UWW8', 'Q86X29', 'Q5SW96', 'Q6PIU2', 'P02647', 'Q9BXJ7', 'O60494', 'Q00341', 'Q0VD83', 'P55056', 'P02654', 'P02649', 'P11150'], 'R-HSA-1474151': ['P31749', 'P31749', 'P29474', 'P07900', 'P0DP23', 'P00374', 'Q03393', 'P35270', 'Q13237', 'P30047', 'P30793'], 'R-HSA-434313': ['P16671', 'P16671'], 'R-HSA-174411': ['P54274', 'P54274', 'Q9BSI4', 'Q15554', 'Q9NYB0', 'Q96AP0', 'Q9NUX5', 'P12004', 'Q9BVC3', 'Q8WVB6', 'P0CG13', 'P35250', 'P40937', 'P35249', 'P40938', 'Q2NKJ3', 'Q9H668', 'Q86WV5', 'P09884', 'P49643', 'P49642', 'Q14181', 'P28340', 'Q9HCU8', 'P49005', 'Q15054'], 'R-HSA-1912399': ['Q9H488', 'Q9H488', 'Q8NBL1'], 'R-HSA-3656253': ['Q93063', 'Q93063'], 'R-HSA-400508': ['P27487', 'P27487', 'P01275', 'Q15005', 'Q9Y6A9', 'Q9BY50', 'P61009', '

4. Map the list of UniprotKB IDs to associated Reactome pathways

In [21]:
dico_pathways_per_protein = dict()
for id in uniprot_ids:
    for pathway, proteins in dico_up_id_per_pathway.items():
        if id in proteins:
            if not id in dico_pathways_per_protein.keys():
                dico_pathways_per_protein[id] = [pathway]
            else:
                dico_pathways_per_protein[id] += [pathway]

print(dico_pathways_per_protein)

{'P16157': ['R-HSA-422475', 'R-HSA-373760', 'R-HSA-447041', 'R-HSA-1266738', 'R-HSA-447038', 'R-HSA-9675108', 'R-HSA-447043'], 'P17542': ['R-HSA-8878171', 'R-HSA-8939236', 'R-HSA-212436', 'R-HSA-74160', 'R-HSA-73857', 'R-HSA-9616222', 'R-HSA-1266738'], 'O60238': ['R-HSA-212436', 'R-HSA-74160', 'R-HSA-73857', 'R-HSA-6803204', 'R-HSA-3700989', 'R-HSA-5633008'], 'P30260': ['R-HSA-453276', 'R-HSA-68867', 'R-HSA-176407', 'R-HSA-141430', 'R-HSA-2559582', 'R-HSA-212436', 'R-HSA-9687139', 'R-HSA-179419', 'R-HSA-176814', 'R-HSA-74160', 'R-HSA-174143', 'R-HSA-8853884', 'R-HSA-1643685', 'R-HSA-73857', 'R-HSA-9675126', 'R-HSA-69239', 'R-HSA-2467813', 'R-HSA-68886', 'R-HSA-176408', 'R-HSA-8953897', 'R-HSA-69052', 'R-HSA-9687136', 'R-HSA-174048', 'R-HSA-68882', 'R-HSA-69017', 'R-HSA-1640170', 'R-HSA-69620', 'R-HSA-69278', 'R-HSA-69242', 'R-HSA-69002', 'R-HSA-179409', 'R-HSA-176409', 'R-HSA-69618', 'R-HSA-69306', 'R-HSA-2559583', 'R-HSA-174178', 'R-HSA-174154', 'R-HSA-2262752', 'R-HSA-176412', 'R-HSA

5. Compute distances between pathways associated to co-expressed proteins

In [ ]:
def compute_distance(abstraction, pathways):
    G = nx.from_pandas_edgelist(df=global_abstraction, source="source", target="target", edge_attr="weight", create_using=nx.DiGraph())
    pathway1 = pathways[0]
    pathway2 = pathways[1]

compute_distance(global_abstraction, ('R-HSA-109582', 'R-HSA-556833'))


[('R-HSA-8877627', 'R-HSA-6806667'), ('R-HSA-6806667', 'R-HSA-196854'), ('R-HSA-6806667', 'R-HSA-2187338'), ('R-HSA-6806664', 'R-HSA-6806667'), ('R-HSA-6806664', 'R-HSA-159740'), ('R-HSA-975634', 'R-HSA-6806667'), ('R-HSA-975634', 'R-HSA-2187338'), ('R-HSA-1296041', 'R-HSA-1296059'), ('R-HSA-1296041', 'R-HSA-997272'), ('R-HSA-1296059', 'R-HSA-1296065'), ('R-HSA-190241', 'R-HSA-5654738'), ('R-HSA-5654738', 'R-HSA-190236'), ('R-HSA-5654696', 'R-HSA-5654738'), ('R-HSA-6803529', 'R-HSA-5654738'), ('R-HSA-5654727', 'R-HSA-5654738'), ('R-HSA-9705683', 'R-HSA-9694516'), ('R-HSA-9694516', 'R-HSA-9679506'), ('R-HSA-9772573', 'R-HSA-9694516'), ('R-HSA-9772572', 'R-HSA-9694516'), ('R-HSA-9672393', 'R-HSA-9662001'), ('R-HSA-9662001', 'R-HSA-9651496'), ('R-HSA-9672391', 'R-HSA-9662001'), ('R-HSA-9672396', 'R-HSA-9662001'), ('R-HSA-9672387', 'R-HSA-9662001'), ('R-HSA-9672395', 'R-HSA-9662001'), ('R-HSA-9672397', 'R-HSA-9662001'), ('R-HSA-9674519', 'R-HSA-9662001'), ('R-HSA-8964208', 'R-HSA-8963691')

In [24]:
subset_proteins = [random.choice(uniprot_ids) for _ in range(5)]
print(subset_proteins)

for prot in subset_proteins:
    if prot in dico_pathways_per_protein.keys():
        associated_pathways = dico_pathways_per_protein[prot]
        pairs = list(combinations(associated_pathways, 2))
        print(pairs)
        

['P20339', 'Q92835', 'P23921', 'P08631', 'Q15532']
[('R-HSA-109582', 'R-HSA-556833'), ('R-HSA-109582', 'R-HSA-1483255'), ('R-HSA-109582', 'R-HSA-1430728'), ('R-HSA-109582', 'R-HSA-983231'), ('R-HSA-109582', 'R-HSA-1660499'), ('R-HSA-109582', 'R-HSA-1483257'), ('R-HSA-556833', 'R-HSA-1483255'), ('R-HSA-556833', 'R-HSA-1430728'), ('R-HSA-556833', 'R-HSA-983231'), ('R-HSA-556833', 'R-HSA-1660499'), ('R-HSA-556833', 'R-HSA-1483257'), ('R-HSA-1483255', 'R-HSA-1430728'), ('R-HSA-1483255', 'R-HSA-983231'), ('R-HSA-1483255', 'R-HSA-1660499'), ('R-HSA-1483255', 'R-HSA-1483257'), ('R-HSA-1430728', 'R-HSA-983231'), ('R-HSA-1430728', 'R-HSA-1660499'), ('R-HSA-1430728', 'R-HSA-1483257'), ('R-HSA-983231', 'R-HSA-1660499'), ('R-HSA-983231', 'R-HSA-1483257'), ('R-HSA-1660499', 'R-HSA-1483257')]
[('R-HSA-202403', 'R-HSA-9680350'), ('R-HSA-202403', 'R-HSA-109582'), ('R-HSA-202403', 'R-HSA-1280218'), ('R-HSA-202403', 'R-HSA-202733'), ('R-HSA-202403', 'R-HSA-168256'), ('R-HSA-202403', 'R-HSA-210990'), ('R